# Reconfigurable LLNN Inference on PYNQ-Z2

This notebook loads the **reconfigurable** LLNN overlay, programs all 4000 gates
with truth tables extracted from the trained model, then runs inference on
binarized 20×20 MNIST test data.

## Prerequisites
Upload these files to `/home/xilinx/reconfig/` on the PYNQ board:
- `llnn.bit` — bitstream (from `make build_reconfig`)
- `llnn.hwh` — hardware handoff (same base name!)
- `weights.json` — truth tables (from `scripts/extract_weights.py`)
- `binarized_20x20_MNIST.txt` — test data (10,000 × 400 binary digits)
- `mnist_test_labels.txt` — labels (one per line, 10,000 lines)

### AXI Address Map (axi_lut_ctrl.sv, 64KB)
| Offset | Description |
|--------|-------------|
| `0x0000–0x3FFC` | Gate programming (write `gate_id * 4` = 32-bit INIT) |
| `0x8000` | STATUS (R) — bit 0 = cfg_busy |
| `0x8004` | TOTAL_GATES (R) |
| `0x9000–0x9030` | NET_I input registers (13 × 32-bit words = 400 bits) |
| `0x9034` | NET_O output register (R) — lower 4 bits = class 0–9 |

## 1. Configuration

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
BITSTREAM_PATH = "/home/xilinx/reconfig/llnn.bit"
WEIGHTS_FILE   = "/home/xilinx/reconfig/weights.json"
MNIST_DATA     = "/home/xilinx/jupyter_notebooks/pynq_notebooks/binarized_20x20_MNIST.txt"
LABELS_FILE    = "/home/xilinx/jupyter_notebooks/pynq_notebooks/mnist_test_labels.txt"

# ── AXI constants (must match axi_lut_ctrl.sv) ─────────────────────────────
ADDR_GATE_BASE   = 0x0000   # Gate programming: gate_id * 4
ADDR_STATUS      = 0x8000   # bit 0 = cfg_busy
ADDR_GATE_COUNT  = 0x8004   # total gates (R)
ADDR_INPUT_BASE  = 0x9000   # 13 × 32-bit input words start here
ADDR_OUTPUT      = 0x9034   # 4-bit classification output
NUM_INPUT_WORDS  = 13       # ceil(400 / 32)
NET_INPUTS       = 400      # 20×20 binarized pixels
AXI_ADDR_SPACE   = 0x10000  # 64 KB

## 2. Load the Overlay

In [ ]:
from pynq import Overlay, MMIO

ol = Overlay(BITSTREAM_PATH)
print("Overlay loaded!")
print("IP dict keys:", list(ol.ip_dict.keys()))

In [ ]:
# Get MMIO handle to our LLNN AXI controller
ip_name = [k for k in ol.ip_dict.keys() if "llnn" in k.lower()]
if not ip_name:
    ip_name = [k for k in ol.ip_dict.keys() if "processing_system" not in k]
ip_name = ip_name[0]

base_addr = ol.ip_dict[ip_name]['phys_addr']
print(f"Using IP: {ip_name}")
print(f"Base address: 0x{base_addr:08X}")

mmio = MMIO(base_addr, AXI_ADDR_SPACE)

## 3. Smoke Test — Read Status Registers

In [ ]:
total_gates = mmio.read(ADDR_GATE_COUNT)
status = mmio.read(ADDR_STATUS)
print(f"TOTAL_GATES = {total_gates} (expected: 4000)")
print(f"STATUS      = 0x{status:08X} (expected: 0 = idle)")
assert total_gates == 4000, f"Expected 4000 gates, got {total_gates}"
print("✅ Smoke test passed!")

## 4. Program All Gates from weights.json

In [ ]:
import json
import time

def wait_ready(mmio, timeout_ms=100):
    """Wait for configuration shift FSM to finish."""
    start = time.time()
    while mmio.read(ADDR_STATUS) & 0x1:
        if (time.time() - start) * 1000 > timeout_ms:
            raise TimeoutError("cfg_busy timeout!")
        time.sleep(0.0001)


def program_gate(mmio, gate_id, init_value):
    """Program a single gate's truth table (32-bit INIT)."""
    wait_ready(mmio)
    mmio.write(ADDR_GATE_BASE + gate_id * 4, init_value & 0xFFFFFFFF)
    wait_ready(mmio)


# Load weights
with open(WEIGHTS_FILE, "r") as f:
    weights = json.load(f)

gates = weights["gates"]
print(f"Loaded {len(gates)} gate truth tables from {WEIGHTS_FILE}")

# Program all gates
t0 = time.time()
for g in gates:
    program_gate(mmio, g["gate_id"], g["init"])
elapsed = time.time() - t0

print(f"✅ Programmed {len(gates)} gates in {elapsed:.2f}s ({len(gates)/elapsed:.0f} gates/sec)")

## 5. Helper Functions

In [ ]:
def write_input(mmio, binary_string):
    """
    Write a 400-character binary string to the 13 AXI input registers.
    Bit 0 of the string maps to bit 0 of register 0 (LSB-first packing).
    """
    bits = [int(c) for c in binary_string[:NET_INPUTS]]
    for w in range(NUM_INPUT_WORDS):
        word = 0
        for b in range(32):
            idx = w * 32 + b
            if idx < len(bits):
                word |= bits[idx] << b
        mmio.write(ADDR_INPUT_BASE + w * 4, word)


def read_output(mmio):
    """Read the 4-bit classification result."""
    return mmio.read(ADDR_OUTPUT) & 0xF


def run_inference(mmio, binary_string):
    """Write input + read output in one call."""
    write_input(mmio, binary_string)
    return read_output(mmio)

## 6. Quick Smoke Test — Write Input + Read Output

In [ ]:
# Write all-zero input
write_input(mmio, "0" * NET_INPUTS)
result = read_output(mmio)
print(f"All-zeros input → class {result}")

# Write all-ones input
write_input(mmio, "1" * NET_INPUTS)
result = read_output(mmio)
print(f"All-ones input  → class {result}")

print("\n✅ AXI communication working!" if result <= 9 else "\n❌ Unexpected output")

## 7. Load MNIST Test Data

In [ ]:
import os

with open(MNIST_DATA, "r") as f:
    samples = [line.strip() for line in f.readlines()]

print(f"Loaded {len(samples)} samples")
print(f"Sample length: {len(samples[0])} bits")

labels = None
if os.path.exists(LABELS_FILE):
    with open(LABELS_FILE, "r") as f:
        labels = [int(line.strip()) for line in f.readlines()]
    print(f"Loaded {len(labels)} labels")
else:
    print("No labels file found.")

## 8. Visualize a Sample

In [ ]:
def show_sample(binary_string, idx=None, label=None, prediction=None):
    """Display a 20×20 binarized image as text art."""
    title = f"Sample {idx}" if idx is not None else "Sample"
    if label is not None:
        title += f" | Label: {label}"
    if prediction is not None:
        match = "✅" if prediction == label else "❌"
        title += f" | Predicted: {prediction} {match}"
    print(title)
    print("─" * 22)
    for row in range(20):
        line = binary_string[row * 20 : (row + 1) * 20]
        print(" " + line.replace("0", "· ").replace("1", "  "))
    print()

# Show first sample
pred = run_inference(mmio, samples[0])
show_sample(samples[0], idx=0, label=labels[0] if labels else None, prediction=pred)

## 9. Run Batch Inference

In [ ]:
NUM_SAMPLES = min(len(samples), 1000)

predictions = []
t0 = time.time()

for i in range(NUM_SAMPLES):
    pred = run_inference(mmio, samples[i])
    predictions.append(pred)

elapsed = time.time() - t0
print(f"Ran {NUM_SAMPLES} inferences in {elapsed:.3f} s")
print(f"Throughput: {NUM_SAMPLES / elapsed:.1f} samples/sec")
print(f"Latency:    {elapsed / NUM_SAMPLES * 1000:.2f} ms/sample")

if labels:
    correct = sum(p == labels[i] for i, p in enumerate(predictions))
    accuracy = correct / NUM_SAMPLES * 100
    print(f"\nAccuracy: {correct}/{NUM_SAMPLES} = {accuracy:.2f}%")

## 10. Confusion Matrix

In [ ]:
if labels:
    matrix = [[0] * 10 for _ in range(10)]
    for i in range(NUM_SAMPLES):
        true_label = labels[i]
        pred_label = predictions[i]
        if true_label < 10 and pred_label < 10:
            matrix[true_label][pred_label] += 1

    print("Confusion Matrix (rows = true, cols = predicted)")
    print("     " + "  ".join(f"{j:3d}" for j in range(10)))
    print("    " + "─" * 45)
    for i in range(10):
        row = "  ".join(f"{matrix[i][j]:3d}" for j in range(10))
        print(f" {i}  │{row}")
else:
    print("Skipping — no labels.")

## 11. Interactive — Classify Specific Samples

In [ ]:
for idx in [0, 1, 2, 3, 4, 100, 500, 999]:
    pred = run_inference(mmio, samples[idx])
    label = labels[idx] if labels else None
    show_sample(samples[idx], idx=idx, label=label, prediction=pred)

## 12. Runtime Reconfiguration Demo

The power of the reconfigurable overlay: reprogram gates **without re-synthesizing**.
Below we demonstrate reprogramming gate 0 and verifying the output changes.

In [ ]:
# Save original output for sample 0
write_input(mmio, samples[0])
original_output = read_output(mmio)
print(f"Original prediction for sample 0: class {original_output}")

# Corrupt gate 0 to all-zeros truth table
program_gate(mmio, 0, 0x00000000)
write_input(mmio, samples[0])
corrupted_output = read_output(mmio)
print(f"After zeroing gate 0: class {corrupted_output}")

# Restore gate 0
original_init = gates[0]["init"]
program_gate(mmio, 0, original_init)
write_input(mmio, samples[0])
restored_output = read_output(mmio)
print(f"After restoring gate 0: class {restored_output}")

assert restored_output == original_output, "Gate restore failed!"
print("\n✅ Runtime reconfiguration verified!")